# RadHarmony: a tour of the dataset API

This notebook shows the main parts of the dataset API on 3 datasets. They are MIMIC-CXR-JPG (2-D JPEG),
CT-RATE (3-D NIfTI) and SIIM-ACR Pneumothorax (2-D DICOM with RLE masks). Each section links to the wiki page
with all the facts.

| Section | Part | Wiki page |
|---|---|---|
| 1 | Harmonizers: from the source files to the harmonized table | [Concepts](https://f10409.github.io/RadHarmony/get-started/concepts.html) |
| 2 | Preprocessors: copies of the images with a fixed size | |
| 3 | Datasets: train and val split, and the samples | [Load a dataset and pick outputs](https://f10409.github.io/RadHarmony/how-to/load-a-dataset.html) |
| 4 | Dataset options: output flags, masks, all rows, folds, cache, image check | [Split data](https://f10409.github.io/RadHarmony/how-to/split-data.html), [Cache the samples](https://f10409.github.io/RadHarmony/how-to/cache.html) |
| 5 | More than one dataset: `ConcatDataset` | |
| 6 | The dataset registry: `list_datasets`, `resolve_dataset`, `@register_dataset` | [Dataset catalog](https://f10409.github.io/RadHarmony/reference/datasets/index.html) |
| 7 | Transforms: image steps and augmentation | [Transforms](https://f10409.github.io/RadHarmony/reference/transforms.html) |

The tour uses the classes of the official train splits, for example `MIMICCXRJPGTrainDataset`. The old classes
that mix the splits, for example `MIMICCXRJPGDataset`, still work. Do not use them for a benchmark
([D-09](https://f10409.github.io/RadHarmony/design.html#d-09-official-splits-are-separate-classes)).

In [ ]:
import matplotlib.pyplot as plt
import monai
import numpy as np
import pandas as pd
import SimpleITK as sitk
from IPython.display import display
from PIL import Image
from torch.utils.data import ConcatDataset

from radharmony.dataset import CTRATETrainDataset, MIMICCXRJPGTrainDataset, SIIMACRPTXTrainDataset
from radharmony.harmonizer import CTRATETrainHarmonizer, MIMICCXRJPGTrainHarmonizer, SIIMACRPTXTrainHarmonizer
from radharmony.preprocessor import CTPreprocessor, CXRPreprocessor
from radharmony.registry import list_datasets, register_dataset, resolve_dataset

# Change these paths to the paths on your computer.
# MIMIC-CXR-JPG
MIMIC_ROOT = "/path/to/mimic-cxr-jpg/2.0.0/"
MIMIC_DIR = MIMIC_ROOT + "files/"
MIMIC_META_CSV = MIMIC_ROOT + "mimic-cxr-2.0.0-metadata.csv"
MIMIC_LABEL_CSV = MIMIC_ROOT + "mimic-cxr-2.0.0-chexpert.csv"
MIMIC_SPLIT_CSV = MIMIC_ROOT + "mimic-cxr-2.0.0-split.csv"
# CT-RATE
CTRATE_DIR = "/path/to/CT-RATE/dataset/train_fixed/"
CTRATE_CSV = "/path/to/CT-RATE/dataset/tables/train_predicted_labels.csv"
CTRATE_META_CSV = "/path/to/CT-RATE/dataset/tables/train_metadata.csv"
# SIIM-ACR Pneumothorax
SIIM_DIR = "/path/to/SIIM_ACR_Pneumothorax/dicom-images-train/"
SIIM_RLE_CSV = "/path/to/SIIM_ACR_Pneumothorax/train-rle.csv"

# The folders that this notebook writes.
MASK_DIR = "./siim_masks/"
MIMIC_PP_DIR = "./mimic_preprocessed/"

# The number of patients to keep from each dataset. None keeps all patients, and the run takes much longer.
N_PATIENTS = 50


def keep_patients(df, n=N_PATIENTS, seed=0):
    """Return the rows of n random patients of a harmonized table. n=None returns all rows."""
    if n is None:
        return df
    ids = df["patient_id"].drop_duplicates().sample(n=n, random_state=seed)
    return df[df["patient_id"].isin(ids)].reset_index(drop=True)

To get the 3 datasets, see [Get the data](https://f10409.github.io/RadHarmony/how-to/get-the-data.html).
MIMIC-CXR-JPG needs a PhysioNet account with the CITI training and the signed data use agreement. CT-RATE
comes from Hugging Face, and SIIM-ACR Pneumothorax comes from Kaggle.

To keep the run short, the tour harmonizes each dataset one time and keeps the images of `N_PATIENTS`
patients. Then it gives this small table to each dataset class with `harmonized_df=`. This is the usual way to
work with a part of a dataset: harmonize first, then filter
([D-02](https://f10409.github.io/RadHarmony/design.html#d-02-harmonize-first-then-filter)).

## 1. Harmonizers

Each `*Harmonizer` class reads the source files of one dataset and returns the **harmonized table**, a
`pd.DataFrame` with one row for each image. All dataset classes use this table. `harmonize()` is enough in
most cases. It can also make copies of the images (section 2) and decode masks (section 1.3).

The table always has these columns:

| Column | Type | Description |
|---|---|---|
| `patient_id` | `str` | The patient ID |
| `study_id` | `str` | The study ID |
| `image_path` | `str` | The path of the image, relative to `base_image_dir` |

The other columns depend on the dataset, for example `view_position`, `mask_path`, `bbox`, `report`,
`age_years`, `sex`, and then the label columns in snake case. See
[Dataset API: Harmonized table columns](https://f10409.github.io/RadHarmony/reference/api-datasets.html#harmonized-table-columns).

**The arguments of `harmonize()`:**

| Argument | Default | Description |
|---|---|---|
| `preprocess_output_dir` | `None` | If set, runs `preprocessor` on the images, writes the copies to this folder and sets `image_path` to the copies (section 2.2) |
| `preprocessor` | `None` | A `CXRPreprocessor` or `CTPreprocessor`. Needed with `preprocess_output_dir` |
| `base_image_dir` | `None` | The folder of the source images, for the copies and the masks. `None` uses the folder of the harmonizer, if it has one |
| `num_workers` | `1` | The number of processes that make the copies |
| `sample_n` | `None` | With `preprocess_output_dir`, makes copies of a random sample of this size, and keeps only these rows. Without it, `sample_n` does nothing and warns |
| `mask_output_dir` | `None` | Decodes the RLE masks to PNG files in this folder, and sets `mask_path` to the PNG files |
| `mask_num_cores` | `1` | The number of threads that decode the masks |
| `study_ids`, `patient_ids` | `None` | Keep only the rows of these IDs. The filter runs early, so the other steps run only for these rows |

### 1.1 MIMIC-CXR-JPG

`MIMICCXRJPGTrainHarmonizer` joins the metadata CSV (`mimic-cxr-2.0.0-metadata.csv`) with the CheXpert label
CSV, and keeps the rows of the official train split (`mimic-cxr-2.0.0-split.csv`). By default it drops the
rows with an uncertain label (`drop_uncertain=True`).

In [ ]:
h_mimic = MIMICCXRJPGTrainHarmonizer(
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    split_csv_path=MIMIC_SPLIT_CSV,
)
df_mimic_all = h_mimic.harmonize()
df_mimic = keep_patients(df_mimic_all)

print(df_mimic_all.shape, "->", df_mimic.shape)
df_mimic.head()

### 1.2 CT-RATE

`CTRATETrainHarmonizer` makes `patient_id`, `study_id` and `image_path` from the `VolumeName` column of the
label CSV. It joins the metadata CSV (`view_position_csv_path`) to add `view_position` (the plane, for example
`axial`), `image_orientation`, `age_years` and `sex`.

| Argument | Description |
|---|---|
| `csv_path` | `train_predicted_labels.csv`: `VolumeName` and 18 label columns |
| `view_position_csv_path` | `train_metadata.csv`. Leave it out to skip the metadata |

- `patient_id` is the first 2 parts of `VolumeName` (split at `_`), and `study_id` is the first 3 parts.
- `image_path` is `{patient}/{study}/{VolumeName}`, relative to `base_image_dir`.

In [ ]:
h_ctrate = CTRATETrainHarmonizer(csv_path=CTRATE_CSV, view_position_csv_path=CTRATE_META_CSV)
df_ctrate_all = h_ctrate.harmonize()
df_ctrate = keep_patients(df_ctrate_all)

print(df_ctrate_all.shape, "->", df_ctrate.shape)
print(df_ctrate.columns.tolist())
df_ctrate.head()

### 1.3 SIIM-ACR Pneumothorax: decode the RLE masks

`SIIMACRPTXTrainHarmonizer` finds all `*.dcm` files below `dicom_dir` to make `image_path`. It makes the label
`pneumothorax` from the `EncodedPixels` column: `-1` gives 0, and a mask gives 1. `mask_path` keeps the RLE
text.

| Argument | Description |
|---|---|
| `csv_path` | `train-rle.csv`, with the columns `ImageId` and `EncodedPixels` |
| `dicom_dir` | The folder of the DICOM files. The harmonizer searches all its subfolders |

With `mask_output_dir`, `harmonize()` decodes the RLE text into one PNG file for each image, and sets
`mask_path` to these files. An image can have more than one line in the CSV, one for each region of the
pneumothorax. The harmonizer joins the masks of all regions into one mask. A second call skips the PNG files
that exist.

To decode only the masks of a few patients, the next cell harmonizes 2 times. The first call finds the
patients. The second call keeps only some of them with `patient_ids=` and decodes their masks.

In [ ]:
h_siim = SIIMACRPTXTrainHarmonizer(csv_path=SIIM_RLE_CSV, dicom_dir=SIIM_DIR)
df_siim_all = h_siim.harmonize()  # mask_path holds the RLE text

ids = None  # None decodes the masks of all images
if N_PATIENTS is not None:
    pos = keep_patients(df_siim_all[df_siim_all["pneumothorax"] == 1], N_PATIENTS // 2)
    neg = keep_patients(df_siim_all[df_siim_all["pneumothorax"] == 0], N_PATIENTS // 2)
    ids = pd.concat([pos, neg])["patient_id"].tolist()
df_siim = h_siim.harmonize(mask_output_dir=MASK_DIR, mask_num_cores=4, patient_ids=ids)
print(df_siim_all.shape, "->", df_siim.shape)

# Show 4 positive and 2 negative images.
positive = df_siim[df_siim["pneumothorax"] == 1].head(4)
negative = df_siim[df_siim["pneumothorax"] == 0].head(2)
samples = pd.concat([positive, negative])
fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4 * len(samples)))
for i, (_, row) in enumerate(samples.iterrows()):
    img = sitk.GetArrayFromImage(sitk.ReadImage(SIIM_DIR + row["image_path"]))
    if img.ndim == 3:
        img = img[0]
    mask = np.array(Image.open(row["mask_path"]))
    label = "positive" if row["pneumothorax"] else "negative"

    axes[i, 0].imshow(img, cmap="gray")
    axes[i, 0].set_title(f"Image ({label})")
    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title("Mask")
    axes[i, 2].imshow(img, cmap="gray")
    axes[i, 2].imshow(mask, alpha=0.4, cmap="Reds")
    axes[i, 2].set_title("Overlay")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()

## 2. Preprocessors

`CXRPreprocessor` and `CTPreprocessor` change the source images to a fixed size and value range. Call a
preprocessor on one file, or give it to `harmonize(preprocess_output_dir=...)` to write a copy of each image.
A dataset class does not need these copies. Its transform reads the source files (section 7).

| Preprocessor | Modality | Input files | Steps | Output |
|---|---|---|---|---|
| `CXRPreprocessor` | 2-D X-ray | DICOM, PNG, JPEG | DICOM only: high-bit fix and VOI LUT. Then: gray float, scale to `[-1, 1]` with the min and max of the image, resize | `SimpleITK.Image` (2-D, float32) |
| `CTPreprocessor` | 3-D CT | NIfTI, NRRD, MHA, MHD (one file, not a folder of DICOM slices) | Gray float, HU window to `[-1, 1]`, `LPS` orientation, crop of the foreground in the axial plane, resize, then the output orientation | `SimpleITK.Image` (3-D, float32) |

**The arguments of both preprocessors:**

| Argument | Type | Default | Description |
|---|---|---|---|
| `target_long_side` | `int` | `112` | The size of the longest side after the resize. For CT, the longest side of the axial plane |
| `interpolation` | SimpleITK interpolator | `sitkLinear` | The interpolator of the resize |

**The other arguments of `CTPreprocessor`:**

| Argument | Type | Default | Description |
|---|---|---|---|
| `window_min` | `float` | `-1000` | The lower HU value of the window |
| `window_max` | `float` | `1000` | The upper HU value of the window |
| `orientation` | `str` | `"LIP"` | The DICOM orientation code of the output |

### 2.1 One image

Call the preprocessor with the path of a file. It returns a `SimpleITK.Image`.

- CXR: the longest side becomes `target_long_side` pixels, and the values go from -1 to 1.
- CT: the longest side of the axial plane becomes `target_long_side` voxels. The z axis gets the same voxel
  spacing as the x and y axes. Example: with `N_PATIENTS = 50`, the next cell reads a CT-RATE volume of 768 ×
  768 × 207 voxels (x, y, z). After the crop, the resize and the LIP orientation, the output has 112 × 101 ×
  94 voxels.

In [ ]:
# CXR: one 2-D JPEG
cxr_prep = CXRPreprocessor(target_long_side=224)
cxr_path = MIMIC_DIR + df_mimic["image_path"].iloc[0]
cxr_out = cxr_prep(cxr_path)
arr_cxr = sitk.GetArrayFromImage(cxr_out)
print(f"CXR  size={cxr_out.GetSize()}  values=[{arr_cxr.min():.3f}, {arr_cxr.max():.3f}]")

# CT: one 3-D NIfTI volume
ct_prep = CTPreprocessor(target_long_side=112, window_min=-1000, window_max=1000)
ct_path = CTRATE_DIR + df_ctrate["image_path"].iloc[0]
ct_raw = sitk.ReadImage(ct_path)
ct_out = ct_prep(ct_path)
arr_ct_raw = sitk.GetArrayFromImage(sitk.Cast(ct_raw, sitk.sitkFloat32)).mean(axis=0)
arr_ct = sitk.GetArrayFromImage(ct_out)
print(f"CT   source size={ct_raw.GetSize()}  size={ct_out.GetSize()}")
print(f"CT   values=[{arr_ct.min():.3f}, {arr_ct.max():.3f}]")

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(sitk.GetArrayFromImage(sitk.ReadImage(cxr_path)), cmap="gray")
axes[0, 0].set_title("CXR: source")
axes[0, 1].imshow(arr_cxr, cmap="gray")
axes[0, 1].set_title("CXR: preprocessed")
axes[1, 0].imshow(arr_ct_raw, cmap="gray")
axes[1, 0].set_title("CT: source (mean projection)")
axes[1, 1].imshow(arr_ct.mean(axis=1), cmap="gray")
axes[1, 1].set_title("CT: preprocessed (mean projection)")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()

### 2.2 All images of a table

Give `preprocess_output_dir` and `preprocessor` to `harmonize()`. The harmonizer writes one `.nii` file for
each image, and sets `image_path` to the path of the copy.

- The copy of `p10/p10000032/s50414267/abc.jpg` is
  `{preprocess_output_dir}/p10/p10000032/s50414267/abc.jpg.nii`. The name keeps the folders and the file
  ending. So `a/1.png` and `a/1.jpg` give 2 files.
- `image_path` is the path of the copy. It is absolute when `preprocess_output_dir` is absolute.
- An `image_path` with a `..` part raises `ValueError`, so each copy stays in `preprocess_output_dir`.
- `sample_n=10` makes copies of 10 random images and keeps only these 10 rows.

A second call writes all files again. It does not skip the files that exist. To skip the work, save the
harmonizer after the first call (`h.save(path)`). Then load it with `load_from_saved(path)` or
`harmonizer_path=`
([Save and reuse a harmonized table](https://f10409.github.io/RadHarmony/how-to/save-and-reuse.html)).

In [ ]:
df_pp = MIMICCXRJPGTrainHarmonizer(
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    split_csv_path=MIMIC_SPLIT_CSV,
).harmonize(
    preprocess_output_dir=MIMIC_PP_DIR,
    preprocessor=CXRPreprocessor(target_long_side=224),
    base_image_dir=MIMIC_DIR,
    num_workers=2,
    sample_n=10,
)

print(df_pp.shape)
print("image_path:", df_pp["image_path"].iloc[0])

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (_, row) in zip(axes.flat, df_pp.iterrows()):
    arr = sitk.GetArrayFromImage(sitk.ReadImage(row["image_path"]))
    ax.imshow(arr, cmap="gray")
    ax.set_title(row["view_position"], fontsize=8)
    ax.axis("off")
plt.suptitle("Preprocessed MIMIC-CXR images")
plt.tight_layout()

In [ ]:
df_pp[["patient_id", "study_id", "image_path", "view_position"]]

## 3. Datasets

A dataset class wraps the harmonized table and a MONAI transform. It gives samples that PyTorch can use. By
default, it uses a MONAI `PersistentDataset`, which keeps the transformed samples on the disk. With
`cache_dir=None`, it uses a MONAI `Dataset` with no disk cache.

`get_datasets()` returns one dataset with all rows. With `n_splits`, it returns
`(train_dataset, val_dataset)`. The split is by patient: no patient is in both splits. See
[Dataset API: Methods](https://f10409.github.io/RadHarmony/reference/api-datasets.html#methods).

The `output_*` flags of the class set the keys of each sample (section 4.1).

### 3.1 CT-RATE

3-D chest CT volumes (NIfTI) with 18 labels from a model. A dataset class takes the source files, or a
harmonized table with `harmonized_df=`. With a table, it does not read the CSV files. The
[CT-RATE page](https://f10409.github.io/RadHarmony/reference/datasets/ct-mri-other/ct_rate.html) lists all
arguments. With no `csv_path`, the class looks for the CSV files in the `tables/` folder next to
`base_image_dir`.

In [ ]:
ctrate_ds_obj = CTRATETrainDataset(
    base_image_dir=CTRATE_DIR,
    harmonized_df=df_ctrate,
    cache_dir="./cache/ctrate",
    output_cls=True,
)
train_ds_ctrate, val_ds_ctrate = ctrate_ds_obj.get_datasets(n_splits=5, num_cores=2)

print(f"CT-RATE   train: {len(train_ds_ctrate):,}  val: {len(val_ds_ctrate):,}")
print(f"Labels ({len(ctrate_ds_obj.LABEL_COLS)}): {ctrate_ds_obj.LABEL_COLS}")

### 3.2 MIMIC-CXR-JPG

2-D chest X-rays (JPEG) with 14 CheXpert labels.

In [ ]:
mimic_ds_obj = MIMICCXRJPGTrainDataset(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    cache_dir="./cache/mimic",
    output_cls=True,
)
train_ds_mimic, val_ds_mimic = mimic_ds_obj.get_datasets(n_splits=5, num_cores=2)

print(f"MIMIC-CXR train: {len(train_ds_mimic):,}  val: {len(val_ds_mimic):,}")
print(f"Labels ({len(mimic_ds_obj.LABEL_COLS)}): {mimic_ds_obj.LABEL_COLS}")

### 3.3 Use only some labels

Set `LABEL_COLS` after you make the dataset, to keep only some labels in the `cls` tensor. `img` and the other
keys do not change.

The `cls` values are part of the cache key. So a new label list writes a new cache entry for each sample, and
the first pass is slow again.

In [ ]:
mimic_ds = MIMICCXRJPGTrainDataset(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    cache_dir="./cache/mimic",
    output_cls=True,
)

In [ ]:
# A new label list: the cache writes new entries for these samples.
mimic_ds.LABEL_COLS = ["atelectasis", "cardiomegaly"]

mimic_subset_ds = mimic_ds.get_datasets(num_cores=2)
sample = mimic_subset_ds[0]

print(f"Dataset size : {len(mimic_subset_ds):,}")
print(f"Labels ({len(mimic_ds.LABEL_COLS)}) : {mimic_ds.LABEL_COLS}")
print(f"cls shape    : {tuple(sample['cls'].shape)}")

### 3.4 SIIM-ACR Pneumothorax

2-D chest X-rays (DICOM) with one label (`pneumothorax`) and segmentation masks from RLE text. The
[SIIM-ACR page](https://f10409.github.io/RadHarmony/reference/datasets/cxr-boxes-masks/siim_acr_ptx.html)
lists all arguments. `output_mask=True` needs `mask_output_dir`, or a table with PNG paths as in section 1.3.

The load step applies the VOI LUT window of each DICOM file
([D-13](https://f10409.github.io/RadHarmony/design.html#d-13-cxr-pixel-values)).

In [ ]:
siim_ds_obj = SIIMACRPTXTrainDataset(
    base_image_dir=SIIM_DIR,
    harmonized_df=df_siim,
    cache_dir="./cache/siim",
    output_cls=True,
)
train_ds_siim, val_ds_siim = siim_ds_obj.get_datasets(n_splits=5, num_cores=4)

print(f"SIIM-ACR  train: {len(train_ds_siim):,}  val: {len(val_ds_siim):,}")
print(f"Labels ({len(siim_ds_obj.LABEL_COLS)}): {siim_ds_obj.LABEL_COLS}")

### 3.5 Look at the samples

Each sample is a `dict` from the MONAI transform. The `output_*` flags set its keys (section 4.1). `img` has
values from -1 to 1. The pad value is -1 for the image and 0 for the mask. `img` is `bfloat16` by default, and
`cls` is always `float32`. See
[Dataset API: Data dict keys](https://f10409.github.io/RadHarmony/reference/api-datasets.html#data-dict-keys).

`visualize_samples` yields one matplotlib `Figure` for each sample. With `per_label=True`, it yields one
figure with a grid of samples. Each plot shows the outputs of the sample:

| Argument | Default | Description |
|---|---|---|
| `n` | `4` | The number of random samples. Not used with `indices` or `per_label=True` |
| `indices` | `None` | The indices of the samples to show |
| `per_label` | `False` | One figure with a grid: one random positive sample for each label |
| `random_state` | `None` | The seed of the random draw |
| `projection_axis` | `0` | The axis of the mean projection of a 3-D volume: `0` axial, `1` coronal, `2` sagittal |

| Key in the sample | What the figure shows |
|---|---|
| `img` (always) | The gray image. A 3-D volume becomes a mean projection along `projection_axis` |
| `cls` | The positive labels, in the title |
| `mask` | A second plot: the image with the mask in red |
| `bbox` | A green box on the image |
| `report` | The report text, below the figure |

In [ ]:
for name, ds_obj, ds in [
    ("CT-RATE", ctrate_ds_obj, train_ds_ctrate),
    ("MIMIC-CXR", mimic_ds_obj, train_ds_mimic),
    ("SIIM-ACR", siim_ds_obj, train_ds_siim),
]:
    s = ds[0]
    positive = [col for col, v in zip(ds_obj.LABEL_COLS, s["cls"].tolist()) if v == 1]
    print(
        f"{name:10s}  img={tuple(s['img'].shape)} {s['img'].dtype}  "
        f"cls={tuple(s['cls'].shape)} {s['cls'].dtype}  positive={positive or ['(none)']}"
    )

In [ ]:
# 2 random samples of each dataset. CT-RATE is 3-D: projection_axis=0 is axial.
# A 2-D dataset does not use projection_axis.
for ds_obj, ds, title in [
    (ctrate_ds_obj, train_ds_ctrate, "CT-RATE"),
    (mimic_ds_obj, train_ds_mimic, "MIMIC-CXR"),
    (siim_ds_obj, train_ds_siim, "SIIM-ACR"),
]:
    print(f"------------------------------ {title} ------------------------------")
    for fig in ds_obj.visualize_samples(ds, n=2, random_state=42, projection_axis=0):
        display(fig)
        plt.close(fig)

# per_label=True: one figure with one plot for each label. SIIM-ACR has 1 label, so the figure has 1 plot.
for fig in siim_ds_obj.visualize_samples(train_ds_siim, per_label=True, random_state=42):
    display(fig)
    plt.close(fig)

## 4. Dataset options

### 4.1 Output flags

The flags of the constructor set the keys of each sample. `img` is always in the sample. A flag that the class
does not support gives a warning and no key
([D-11](https://f10409.github.io/RadHarmony/design.html#d-11-output-flags)).

The 5 flags are `output_cls`, `output_mask`, `output_report`, `output_bbox` and `output_reg`. All are `False`
by default.
[Dataset API: Data dict keys](https://f10409.github.io/RadHarmony/reference/api-datasets.html#data-dict-keys)
gives the keys, shapes and types of each flag.

With no flag, the dataset returns `{"img": tensor}` only, for example for self-supervised training or for
inference.

In [ ]:
img_only_ds = SIIMACRPTXTrainDataset(
    base_image_dir=SIIM_DIR,
    harmonized_df=df_siim,
    cache_dir=None,
    output_cls=False,
).get_datasets()

print("output_cls=False ->", list(img_only_ds[0].keys()))

### 4.2 Segmentation masks (`output_mask=True`)

`output_mask=True` adds a `mask` tensor to each sample. The transform loads the file of `mask_path`, resizes
it to the size of `img` with nearest-neighbor interpolation, and pads it with 0. The PNG files hold 0 and 255.
The sample holds 0 and 1.

For SIIM-ACR, give `mask_output_dir` to the dataset class, or a table with PNG paths as in section 1.3. An
image with an empty `mask_path` has no annotation, and `output_mask=True` drops it
([D-33](https://f10409.github.io/RadHarmony/design.html#d-33-an-empty-mask_path-means-not-annotated)).

| Key | Shape | dtype | Description |
|---|---|---|---|
| `img` | `(1, H, W)` | `bfloat16` | The X-ray, from -1 to 1 |
| `cls` | `(1,)` | `float32` | The label `pneumothorax` |
| `mask` | `(1, H, W)` | `float32` | The mask, 0 or 1 |

In [ ]:
mask_ds = SIIMACRPTXTrainDataset(
    base_image_dir=SIIM_DIR,
    harmonized_df=df_siim,
    cache_dir=None,
    output_cls=True,
    output_mask=True,
).get_datasets()

sample_mask = mask_ds[0]
print("Keys :", list(sample_mask.keys()))
print("img  :", sample_mask["img"].shape, sample_mask["img"].dtype)
print("mask :", sample_mask["mask"].shape, sample_mask["mask"].dtype, sample_mask["mask"].unique().tolist())
print("cls  :", sample_mask["cls"].tolist())

### 4.3 All rows (no train and val split)

With no `n_splits`, `get_datasets()` returns **one dataset** with all patients. Use it to run a model on all
images, to get features, or when you make your own split.

```python
full_ds = SomeDataset(...).get_datasets()  # no n_splits: one dataset
train_ds, val_ds = SomeDataset(...).get_datasets(n_splits=10)  # about 10% of the patients go to val
```

In [ ]:
full_ds_mimic = MIMICCXRJPGTrainDataset(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    cache_dir="./cache/mimic_full",
    output_cls=True,
).get_datasets()

print(f"MIMIC-CXR all rows: {len(full_ds_mimic):,}")

### 4.4 K-fold cross-validation

`get_folds(n_splits=N)` yields one `(train_dataset, val_dataset)` tuple for each fold. It shuffles the
patients one time and puts them in `N` groups of about the same size. Each fold uses one group as the val
split. See [Dataset API: Methods](https://f10409.github.io/RadHarmony/reference/api-datasets.html#methods) for
its arguments.

Each fold has its own cache folders (`fold_<i>/train` and `fold_<i>/val`). So all folds can be in the cache at
the same time. See
[Split data: Make k folds](https://f10409.github.io/RadHarmony/how-to/split-data.html#3-make-k-folds).

In [ ]:
folds_ds = MIMICCXRJPGTrainDataset(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    cache_dir="./cache/mimic_folds",
    output_cls=True,
)
for fold, (train_fold, val_fold) in enumerate(folds_ds.get_folds(n_splits=5, num_cores=4)):
    print(f"Fold {fold}: train={len(train_fold):,}  val={len(val_fold):,}")

### 4.5 No cache (`cache_dir=None`)

With `cache_dir=None`, the dataset uses a MONAI `Dataset` in place of a `PersistentDataset`. It writes no
files, and it runs the transform each time that you read a sample.

**When to use it:**

- A quick test, where the cache files are not worth the time.
- A computer with little disk space.
- A transform that is fast, so the cache does not help.

Each pass runs the transform again. With a cache, only the first pass runs the steps before the augmentation.
The augmentation steps run each time. See
[Cache the samples](https://f10409.github.io/RadHarmony/how-to/cache.html).

In [ ]:
train_nc, val_nc = MIMICCXRJPGTrainDataset(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    cache_dir=None,
    output_cls=True,
).get_datasets(n_splits=5, num_cores=4)

print(isinstance(train_nc, monai.data.Dataset), isinstance(train_nc, monai.data.PersistentDataset))

### 4.6 Fill the cache

`pre_cache(dataset)` reads each sample one time, so that the `PersistentDataset` writes it to the disk. Then
the next reads load the sample from the disk and do not run the steps before the augmentation again. The
augmentation steps still run each time. Call it before the training, so that the first pass is not slow.

Give it a split from `get_datasets()` or `get_folds()`. See
[Cache the samples](https://f10409.github.io/RadHarmony/how-to/cache.html).

The cache uses a hash of the input dict of each sample. A new entry in the dict (for example `mask`) or a new
`LABEL_COLS` gives new cache files. The name of the cache folder also has a short hash of the steps before the
augmentation (load, scale, resize, pad). So a new `img_size` makes a new cache folder, but a new augmentation
setting does not ([D-15](https://f10409.github.io/RadHarmony/design.html#d-15-cache-before-augmentation)).

In [ ]:
siim_ds = SIIMACRPTXTrainDataset(
    base_image_dir=SIIM_DIR,
    harmonized_df=df_siim,
    cache_dir="./cache/siim_precache",
    output_cls=True,
)
train_ds, val_ds = siim_ds.get_datasets(n_splits=5, num_cores=4)

# Fill the cache before the training. The next reads load the samples from the disk.
siim_ds.pre_cache(train_ds, num_workers=4)
siim_ds.pre_cache(val_ds, num_workers=4)

### 4.7 Check the image files (`verify_images`)

`verify_images()` checks that the file of each `image_path` is on the disk. With `drop_missing=True` (the
default), it drops the rows with no file, so the next `get_datasets()` calls do not use them. With
`drop_missing=False`, it returns these rows and does not drop them.

Both the dataset class and the harmonizer have `verify_images()`. The harmonizer uses its own folder, or the
`base_image_dir` that you give.

In [ ]:
# Dataset class: it uses base_image_dir.
ds = MIMICCXRJPGTrainDataset(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    cache_dir=None,
    output_cls=True,
)
missing = ds.verify_images()  # drop_missing=True by default
print(f"Missing files: {len(missing)}")
train_ds, val_ds = ds.get_datasets(n_splits=5)
print(f"train: {len(train_ds):,}  val: {len(val_ds):,}")

# Harmonizer: h_siim uses its dicom_dir. It checks the rows of its last harmonize() call.
missing = h_siim.verify_images(drop_missing=False)
print(f"Missing files: {len(missing)}")

## 5. More than one dataset

`torch.utils.data.ConcatDataset` joins some datasets into one dataset, with one index for all samples. Use it
to train on data from more than one source.

- All datasets must give samples with the **same keys** (the `output_*` flags) and shapes that can go in one
  batch. For example, a 2-D and a 3-D `img` cannot go in one batch with the default `DataLoader` collate.
- With `output_cls=True`, the `cls` tensors must have the **same length** in all datasets. If not, write your
  own collate function. In the next cell, MIMIC-CXR has 14 labels and SIIM-ACR has 1, so a `DataLoader` cannot
  batch the joined dataset with the default collate.
- Each dataset keeps its own cache. The joined dataset has no shared cache.

In [ ]:
combined = ConcatDataset([train_ds_mimic, train_ds_siim])

print(f"MIMIC-CXR train : {len(train_ds_mimic):,}")
print(f"SIIM-ACR  train : {len(train_ds_siim):,}")
print(f"Combined        : {len(combined):,}")

## 6. Dataset registry

The registry maps a name (the registry key) to each dataset class. Section 5 of
[`radharmony_hands_on.ipynb`](radharmony_hands_on.ipynb) shows `list_datasets()` and `resolve_dataset()`. The
[Dataset catalog](https://f10409.github.io/RadHarmony/reference/datasets/index.html) lists all keys. This
section shows 2 more uses: a dataset from a config, and a dataset class of your own.

### 6.1 Make a dataset from a config

With the registry, the dataset name can come from a config dict or a YAML file. Then the training script needs
no import of a dataset class.

In [ ]:
# A config, as it can come from YAML, argparse or Hydra.
cfg = {
    "dataset": "mimic_cxr_jpg_train",
    "base_image_dir": MIMIC_DIR,
    "harmonized_df": df_mimic,  # for a full run: csv_path, label_csv_path and split_csv_path
    "cache_dir": "./cache/mimic_registry",
    "output_cls": True,
}

DatasetCls = resolve_dataset(cfg.pop("dataset"))  # name -> class
train_ds, val_ds = DatasetCls(**cfg).get_datasets(n_splits=5, num_cores=4)

print(f"train: {len(train_ds):,}  val: {len(val_ds):,}")
print("Sample keys:", list(train_ds[0].keys()))

### 6.2 Register a dataset of your own

Put `@register_dataset("key")` on a subclass of `BaseRadiologicalDataset` or of a dataset class of the
package. The class goes into the registry when the cell runs. Then it works like the classes of the package:
`list_datasets()` shows it, and `resolve_dataset()` returns it.

The next cell keeps only the PA views of the MIMIC-CXR-JPG train split. It filters in `_get_harmonized_df()`,
after the parent class gives the table. So the filter also works on a table from `harmonized_df=` or
`harmonizer_path=`.

A second run of the cell raises `ValueError`, because a key can be in the registry only one time (see step 6
of [`custom_dataset_tutorial.ipynb`](custom_dataset_tutorial.ipynb)). That notebook also shows how to add a
new dataset from its source files.

In [ ]:
@register_dataset("mimic_cxr_jpg_train_pa")
class MIMICCXRJPGTrainPADataset(MIMICCXRJPGTrainDataset):
    """MIMIC-CXR-JPG train split, PA views only."""

    def _get_harmonized_df(self):
        df = super()._get_harmonized_df()
        return df[df["view_position"] == "PA"].reset_index(drop=True)


print("mimic_cxr_jpg_train_pa" in list_datasets())

DatasetCls = resolve_dataset("mimic_cxr_jpg_train_pa")
pa_ds = DatasetCls(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    cache_dir="./cache/mimic_pa",
    output_cls=True,
).get_datasets()

print(f"All views: {len(df_mimic):,}  PA only: {len(pa_ds):,}")

## 7. Transform builder

`RadiologyTransform2D` and `RadiologyTransform3D` build the image steps and the augmentation of a dataset. The
order is always: **load, scale, resize, pad, augment, convert to tensor, keep the output keys**.

The `with_*` methods add augmentation steps. Most of them take `prob`, and each returns the builder, so you
can chain the calls. `.get_transform()` at the end returns a `monai.transforms.Compose`. See
[Transforms](https://f10409.github.io/RadHarmony/reference/transforms.html) for all steps, methods and
defaults.

### 7.1 Image steps only (no augmentation)

The next cell gives a transform with the image steps only. This is the same as the default transform of the
dataset, so you can also leave out `transform=`.

In [ ]:
from radharmony.dataset import RadiologyTransform2D

# The image steps only: the same as the default transform of the dataset.
transform = RadiologyTransform2D(img_size=224, output_keys={"img", "cls"}).get_transform()

ds = MIMICCXRJPGTrainDataset(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    transform=transform,
    cache_dir="./cache/mimic_transforms",
    output_cls=True,  # output_keys of the transform must have "cls"
)
train_ds, val_ds = ds.get_datasets(n_splits=5, num_cores=4)

sample = train_ds[0]
print("Keys  :", list(sample.keys()))
print("Image :", sample["img"].shape, sample["img"].dtype)
print("Labels:", sample["cls"].shape)

### 7.2 Other transforms for the train and val splits

Give `train_transform=` and `val_transform=` to `get_datasets()` or `get_folds()`. Put the augmentation in the
train transform only. If you give no `val_transform` and the dataset transform has a random step, the call
warns ([D-49](https://f10409.github.io/RadHarmony/design.html#d-49-a-val-split-with-random-steps-warns)).

In [ ]:
train_transform = (
    RadiologyTransform2D(img_size=224, output_keys={"img", "cls"})
    .with_flip(spatial_axis=1, prob=0.5)  # left-right flip
    .with_affine(translate_range=(10, 10), rotate_range=(0.17,), scale_range=(0.1, 0.1), prob=0.5)
    .with_intensity_jitter(shift=0.1, scale=0.1)
    .with_gaussian_noise(std=0.03)
    .get_transform()
)
# No augmentation for the val split.
val_transform = RadiologyTransform2D(img_size=224, output_keys={"img", "cls"}).get_transform()

ds = MIMICCXRJPGTrainDataset(
    base_image_dir=MIMIC_DIR,
    harmonized_df=df_mimic,
    cache_dir="./cache/mimic_aug",
    output_cls=True,
)
train_ds, val_ds = ds.get_datasets(
    n_splits=5,
    num_cores=4,
    train_transform=train_transform,
    val_transform=val_transform,
)

In [ ]:
N = 5  # images in each row

fig, axes = plt.subplots(2, N, figsize=(3 * N, 7))
for col in range(N):
    for row, (split, title) in enumerate([(train_ds, "Train (augmented)"), (val_ds, "Val (no augmentation)")]):
        # (1, H, W) bfloat16 from -1 to 1  ->  (H, W) float32 from 0 to 1
        img = split[col]["img"].float().squeeze(0).numpy()
        img = np.clip((img + 1.0) / 2.0, 0.0, 1.0)

        ax = axes[row, col]
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
        if col == 0:
            ax.set_title(title, fontsize=10, loc="left", pad=4)

plt.suptitle("Train and val: the first 5 samples of each split", y=1.01)
plt.tight_layout()
plt.show()

### 7.3 2-D with a mask (SIIM-ACR)

With `output_mask=True`, put `"mask"` in `output_keys`. Then the transform applies the resize, flip and affine
steps to the mask in the same way as to the image.

In [ ]:
transform = (
    RadiologyTransform2D(img_size=512, output_keys={"img", "cls", "mask"})
    .with_flip(spatial_axis=1, prob=0.5)
    .with_affine(translate_range=(15, 15), scale_range=(0.1, 0.1), prob=0.5)
    # spacing: the grid step in pixels. Keep the magnitude much smaller than the spacing.
    .with_elastic_deformation(spacing=100, magnitude_range=(5, 15), prob=0.2)
    .get_transform()
)

ds = SIIMACRPTXTrainDataset(
    base_image_dir=SIIM_DIR,
    harmonized_df=df_siim,
    transform=transform,
    cache_dir="./cache/siim_mask",
    output_cls=True,
    output_mask=True,
)
plain = RadiologyTransform2D(img_size=512, output_keys={"img", "cls", "mask"}).get_transform()
train_ds, val_ds = ds.get_datasets(n_splits=5, num_cores=4, val_transform=plain)  # no random step in val

sample = train_ds[0]
print("Keys :", list(sample.keys()))
print("Image:", sample["img"].shape)
print("Mask :", sample["mask"].shape)

In [ ]:
for fig in ds.visualize_samples(train_ds, n=2, random_state=0):
    display(fig)
    plt.close(fig)

### 7.4 3-D CT with an HU window (CT-RATE)

`hu_window=(min_HU, max_HU)` clips the HU values before the value scaling. The 3-D `with_elastic_deformation`
takes `sigma_range` (how smooth the warp is), not `spacing`.

In [ ]:
from radharmony.dataset import RadiologyTransform3D

transform = (
    RadiologyTransform3D(img_size=112, output_keys={"img", "cls"}, hu_window=(-1000, 400))  # a lung window
    .with_flip(spatial_axis=0, prob=0.5)
    .with_affine(translate_range=(5, 5, 5), rotate_range=(0.09, 0.09, 0.09), prob=0.5)
    .with_gaussian_smooth(sigma_range=(0.5, 1.0), prob=0.2)
    .with_elastic_deformation(sigma_range=(5.0, 7.0), magnitude_range=(50, 150), prob=0.2)
    .get_transform()
)

ds = CTRATETrainDataset(
    base_image_dir=CTRATE_DIR,
    harmonized_df=df_ctrate,
    transform=transform,
    cache_dir="./cache/ct_rate_aug",
    output_cls=True,
)
train_ds = ds.get_datasets()  # one dataset: this cell shows only the train transform

sample = train_ds[0]
print("Keys  :", list(sample.keys()))
print("Volume:", sample["img"].shape, sample["img"].dtype)
print("Labels:", sample["cls"].shape)

### 7.5 Add any MONAI transform

`.add_transform()` adds a MONAI transform that has no `with_*` method. If the sample has a box (`bbox`), this
step does not change the box to match the image.

In [ ]:
transform = (
    RadiologyTransform2D(img_size=224, output_keys={"img"})
    .with_flip(spatial_axis=1)
    .add_transform(  # any MONAI transform
        monai.transforms.RandAffineD(keys=["img"], prob=0.3, translate_range=(10, 10), mode="bilinear")
    )
    .get_transform()
)

print(type(transform))
print(f"{len(transform.transforms)} steps in the transform")